# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR² colorectal cancer dataset using the `mlcroissant` library and pandas in Python.

The dataset contains clinical and pathological data on 77 cancer survivors with second primary colorectal cancer, with variables such as demographics, comorbidities, treatments, anatomical location, histopathological subtype, distant metastasis, and microsatellite instability (MSI) status.

----
### Dataset Source
- **Croissant schema URL**: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Install mlcroissant if not already present
!pip install --quiet mlcroissant

## 1. Data Loading

Load the Croissant metadata and explore basic dataset information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f'Dataset: {metadata.name}\n')
print(f'Description: {metadata.description}\n')
print(f"Identifier: {metadata.identifier}")
print(f"Published on: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review available record sets and their fields in the package, referencing each by their `@id`. The Croissant format organizes tabular data into *record sets*, each with named fields (columns).

Let's list all record sets, along with their corresponding field `@id`s.

In [ ]:
# Get all record sets from the dataset
record_sets = dataset.record_sets

if record_sets:
    for i, rs in enumerate(record_sets):
        print(f"Record Set {i+1}:")
        print(f"  Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Description: {rs.description if rs.description else 'N/A'}")
        print(f"  Fields (by @id):")
        for fld in rs.fields:
            print(f"    - {fld.id} (name: {fld.name}, dataType: {fld.data_type})")
        print()
else:
    print("No record sets found in this dataset. Please confirm the schema structure.")

## 3. Data Extraction

Let's extract records from the primary record set into pandas DataFrames for further analysis. All references use the `@id` of the entities.

In [ ]:
# Extract all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# For demonstration, extract all recordsets (there may be only one main table)
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f'Record Set: {rs_id}, Rows: {len(df)}, Columns: {list(df.columns)}')

# Display the head of the main record set DataFrame
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Common data analysis includes filtering, normalization, and grouped summaries. In this example, we'll:
- Select a numeric field by `@id` (e.g., age at diagnosis or years-of-interval if present),
- Filter patients above a threshold,
- Normalize the variable,
- Group by another field (e.g., sex or tumor location by `@id`).

> **Note:** Update the field `@id`s if needed after inspecting available fields above.

In [ ]:
# Adapt these IDs to match what is present in the record set above:
# Example IDs (replace with actual @ids listed for your dataset):
numeric_field_id = None
group_field_id = None
main_df = dataframes.get(main_rs_id)

# Attempt to auto-select likely numeric fields by name heuristics
if main_df is not None:
    for col in main_df.columns:
        if any(x in col.lower() for x in ['age', 'years', 'interval', 'duration', 'count', 'number']):
            numeric_field_id = col
            break
    if not numeric_field_id:
        # Fallback: pick first float/int column
        for col in main_df.select_dtypes(include=['float64', 'int64']).columns:
            numeric_field_id = col
            break

    # Attempt to pick a groupable categorical field (e.g. sex, location)
    for col in main_df.columns:
        if any(x in col.lower() for x in ['sex', 'gender', 'location', 'group', 'type']):
            group_field_id = col
            break

    if not numeric_field_id:
        print("No obvious numeric field was found to demonstrate EDA.")
    else:
        # Ensure numeric
        pd.options.mode.chained_assignment = None
        main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
        threshold = main_df[numeric_field_id].mean() if pd.notnull(main_df[numeric_field_id].mean()) else 10

        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df[[numeric_field_id] + ([group_field_id] if group_field_id else [])].head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id and group_field_id in filtered_df:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std']).sort_values('count', ascending=False)
            print(f"Grouped statistics by '{group_field_id}':")
            display(grouped_df)
else:
    print("No data available for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relation to the grouping field if set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_df is not None and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15, color="teal")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    # Boxplot by group
    if group_field_id and group_field_id in main_df:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id], palette="Set2")
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- Demonstrated extraction of structured clinical oncology data from a FAIR²-compliant Croissant schema.
- Reviewed available record sets and their fields using `@id` references for reproducibility.
- Extracted data into pandas and performed basic filtering, normalization, grouping, and visualization.

This notebook provides a reproducible workflow for working with Croissant datasets in Python and can be extended for modeling or in-depth domain analysis.